# 01 · YouTube audio extraction

**Portfolio adaptation of the 2023 SJK workflow:** discover/download public YouTube audio, keep a structured video metadata CSV, convert the audio to MP3 for speech transcription.

The original sanitized notebook is in `notebooks/sanitized_reference/Step5_Scrape_Youtube.ipynb`. This featured edition repairs portable paths and separates file processing from live YouTube access. **It defaults to a synthetic, offline demonstration.** No content is downloaded from YouTube unless explicitly enabled. External libraries/streams can change.

## Setup
The demonstration uses Python's standard library and `ffmpeg`. Google Colab normally includes `ffmpeg`; elsewhere install it separately. For real YouTube downloads, install `pytube` first (e.g. `%pip install pytube`). Live scraping/downloading requires lawful permission and respect for site terms.


In [ ]:
from pathlib import Path
from datetime import date
import csv
import os
import subprocess

PROJECT_DIR = Path(os.environ.get("SJK_PORTFOLIO_DIR", "/content/sjk_portfolio"))
AUDIO_DIR = PROJECT_DIR / "audio"
MP3_DIR = AUDIO_DIR / "mp3_converted"
METADATA_CSV = AUDIO_DIR / "metadata.csv"
DEMO_MODE = os.environ.get("SJK_DEMO_MODE", "1") == "1"
for directory in (AUDIO_DIR, MP3_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Workspace:", PROJECT_DIR, "Demo mode:", DEMO_MODE)


## Video metadata and MP4-to-MP3 conversion
These operations preserve the essential original pipeline (video ID → metadata → audio → converted MP3), with repeatable file handling and clear errors.


In [ ]:
METADATA_COLUMNS = ["Title", "Video ID", "Channel ID", "Description", "Length (sec)", "Date Uploaded"]

def save_metadata(rows, csv_file=METADATA_CSV):
    """Append new video records by ID; avoid duplicate metadata on reruns."""
    csv_file = Path(csv_file)
    csv_file.parent.mkdir(parents=True, exist_ok=True)
    previous = []
    if csv_file.exists():
        with csv_file.open(newline="", encoding="utf-8") as f:
            previous = list(csv.DictReader(f))
    found = {row.get("Video ID") for row in previous}
    additions = []
    for row in rows:
        video_id = row["Video ID"]
        if video_id not in found:
            additions.append(row)
            found.add(video_id)
    with csv_file.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=METADATA_COLUMNS)
        writer.writeheader()
        writer.writerows(previous + additions)
    return len(additions)


def convert_mp4_to_mp3(mp4_file, mp3_dir=MP3_DIR):
    """Convert a local MP4 audio file without scraping anything."""
    mp4_file = Path(mp4_file)
    mp3_dir = Path(mp3_dir)
    mp3_dir.mkdir(parents=True, exist_ok=True)
    target = mp3_dir / (mp4_file.stem + ".mp3")
    if not mp4_file.is_file():
        raise FileNotFoundError(mp4_file)
    subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-y",
                    "-i", str(mp4_file), str(target)], check=True)
    return target


def download_public_audio(url, audio_dir=AUDIO_DIR):
    """Opt-in network step modeled on original pytube logic; not used in demo."""
    from pytube import Playlist, YouTube  # optional, may require version updates
    if "list=" in url:
        videos = Playlist(url).videos
    else:
        videos = [YouTube(url)]
    rows = []
    for video in videos:
        destination = Path(audio_dir) / (video.video_id + ".mp4")
        if not destination.exists():
            stream = video.streams.filter(only_audio=True, subtype="mp4").first()
            if stream is None:
                raise RuntimeError("No compatible audio stream available")
            stream.download(output_path=str(audio_dir), filename=destination.name)
        rows.append({"Title": video.title, "Video ID": video.video_id,
                     "Channel ID": video.channel_id, "Description": video.description,
                     "Length (sec)": video.length, "Date Uploaded": str(video.publish_date)})
    save_metadata(rows, Path(audio_dir) / "metadata.csv")
    return [Path(audio_dir) / (video["Video ID"] + ".mp4") for video in rows]


## Demonstration
Demo mode generates a ~0.3-second synthetic tone as an MP4 and converts it using the same function as real inputs. It is **not** evidence of a live YouTube API/download test.


In [ ]:
if DEMO_MODE:
    sample_id = "synthetic001"
    sample_mp4 = AUDIO_DIR / f"{sample_id}.mp4"
    subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-y",
                    "-f", "lavfi", "-i", "sine=frequency=440:duration=0.3",
                    "-c:a", "aac", str(sample_mp4)], check=True)
    added = save_metadata([{"Title": "Synthetic sample tone", "Video ID": sample_id,
                            "Channel ID": "example", "Description": "Generated for portfolio testing",
                            "Length (sec)": "0.3", "Date Uploaded": "N/A"}])
    sample_mp3 = convert_mp4_to_mp3(sample_mp4)
    print("Demo metadata rows added:", added)
    print("Converted sample:", sample_mp3, "size:", sample_mp3.stat().st_size)
else:
    video_url = os.environ.get("SJK_YOUTUBE_URL")
    if not video_url:
        print("To enable a live run, set SJK_YOUTUBE_URL and SJK_DEMO_MODE=0.")
    else:
        converted = [convert_mp4_to_mp3(item) for item in download_public_audio(video_url)]
        print("Converted", len(converted), "files")


## Limits
A real YouTube download was **not** run in this portfolio test. YouTube layouts, pytube stream availability, site terms and permissions may change. Input files and generated audio are intentionally excluded from Git.
